<a href="https://colab.research.google.com/github/mrdbourke/pytorch-deep-learning/blob/main/extras/exercises/07_pytorch_experiment_tracking_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. PyTorch Experiment Tracking Exercise Template

Welcome to the 07. PyTorch Experiment Tracking exercise template notebook.

> **Note:** There may be more than one solution to each of the exercises. This notebook only shows one possible example.

## Resources

1. These exercises/solutions are based on [section 07. PyTorch Transfer Learning](https://www.learnpytorch.io/07_pytorch_experiment_tracking/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.
2. See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/cO_r2FYcAjU).
3. See [other solutions on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/extras/solutions).

> **Note:** The first section of this notebook is dedicated to getting various helper functions and datasets used for the exercises. The exercises start at the heading "Exercise 1: ...".

### Get various imports and helper functions

We'll need to make sure we have `torch` v.1.12+ and `torchvision` v0.13+.

In [1]:
# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install -U --pre torch torchvision --extra-index-url https://download.pytorch.org/whl/nightly/cu113
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")

torch version: 2.14.0+cpu
torchvision version: 0.29.0+cpu


In [2]:
# Make sure we have a GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [3]:
# Get regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    %pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory
try:
    from going_modular.going_modular import data_setup, engine
except ImportError:
    # This repo already has going_modular/ two directories up (../../going_modular),
    # so point Python at the repo root instead of git-cloning the whole repo again.
    # Note: the rest of this notebook (e.g. the custom train() below) imports things
    # like "going_modular.going_modular.engine", so the repo ROOT (not the
    # going_modular folder itself) needs to be on sys.path for that dotted path to work.
    import sys
    from pathlib import Path
    repo_root = Path.cwd()
    while not (repo_root / "going_modular").is_dir() and repo_root != repo_root.parent:
        repo_root = repo_root.parent
    if (repo_root / "going_modular").is_dir():
        print(f"[INFO] Found going_modular at {repo_root / 'going_modular'}, adding {repo_root} to sys.path.")
        sys.path.insert(0, str(repo_root))
        from going_modular.going_modular import data_setup, engine
    else:
        print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
        !git clone https://github.com/mrdbourke/pytorch-deep-learning
        !mv pytorch-deep-learning/going_modular .
        !rm -rf pytorch-deep-learning
        from going_modular.going_modular import data_setup, engine

[INFO] Found going_modular at D:\computer vision\baith02_cv_DoanThiThuLinh\pytorch-deep-learning\going_modular, adding D:\computer vision\baith02_cv_DoanThiThuLinh\pytorch-deep-learning to sys.path.


In [4]:
# Set seeds
def set_seeds(seed: int=42):
    """Sets random sets for torch operations.

    Args:
        seed (int, optional): Random seed to set. Defaults to 42.
    """
    # Set the seed for general torch operations
    torch.manual_seed(seed)
    # Set the seed for CUDA torch operations (ones that happen on the GPU)
    torch.cuda.manual_seed(seed)

In [5]:
import os
import zipfile

from pathlib import Path

import requests

def download_data(source: str, 
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                      destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

[INFO] data\pizza_steak_sushi directory exists, skipping download.


WindowsPath('data/pizza_steak_sushi')

In [6]:
from torch.utils.tensorboard import SummaryWriter
def create_writer(experiment_name: str, 
                  model_name: str, 
                  extra: str=None):
    """Creates a torch.utils.tensorboard.writer.SummaryWriter() instance saving to a specific log_dir.

    log_dir is a combination of runs/timestamp/experiment_name/model_name/extra.

    Where timestamp is the current date in YYYY-MM-DD format.

    Args:
        experiment_name (str): Name of experiment.
        model_name (str): Name of model.
        extra (str, optional): Anything extra to add to the directory. Defaults to None.

    Returns:
        torch.utils.tensorboard.writer.SummaryWriter(): Instance of a writer saving to log_dir.

    Example usage:
        # Create a writer saving to "runs/2022-06-04/data_10_percent/effnetb2/5_epochs/"
        writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb2",
                               extra="5_epochs")
        # The above is the same as:
        writer = SummaryWriter(log_dir="runs/2022-06-04/data_10_percent/effnetb2/5_epochs/")
    """
    from datetime import datetime
    import os

    # Get timestamp of current date (all experiments on certain day live in same folder)
    timestamp = datetime.now().strftime("%Y-%m-%d") # returns current date in YYYY-MM-DD format

    if extra:
        # Create log directory path
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)
        
    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)

In [7]:
# Create a test writer
writer = create_writer(experiment_name="test_experiment_name",
                       model_name="this_is_the_model_name",
                       extra="add_a_little_extra_if_you_want")

[INFO] Created SummaryWriter, saving to: runs\2026-09-24\test_experiment_name\this_is_the_model_name\add_a_little_extra_if_you_want...


In [8]:
from typing import Dict, List
from tqdm.auto import tqdm

from going_modular.going_modular.engine import train_step, test_step

# Add writer parameter to train()
def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device, 
          writer: torch.utils.tensorboard.writer.SummaryWriter # new parameter to take in a writer
          ) -> Dict[str, List]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Stores metrics to specified writer log_dir if present.

    Args:
      model: A PyTorch model to be trained and tested.
      train_dataloader: A DataLoader instance for the model to be trained on.
      test_dataloader: A DataLoader instance for the model to be tested on.
      optimizer: A PyTorch optimizer to help minimize the loss function.
      loss_fn: A PyTorch loss function to calculate loss on both datasets.
      epochs: An integer indicating how many epochs to train for.
      device: A target device to compute on (e.g. "cuda" or "cpu").
      writer: A SummaryWriter() instance to log model results to.

    Returns:
      A dictionary of training and testing loss as well as training and
      testing accuracy metrics. Each metric has a value in a list for 
      each epoch.
      In the form: {train_loss: [...],
                train_acc: [...],
                test_loss: [...],
                test_acc: [...]} 
      For example if training for epochs=2: 
              {train_loss: [2.0616, 1.0537],
                train_acc: [0.3945, 0.3945],
                test_loss: [1.2641, 1.5706],
                test_acc: [0.3400, 0.2973]} 
    """
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

        # Print out what's happening
        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)


        ### New: Use the writer parameter to track experiments ###
        # See if there's a writer, if so, log to it
        if writer:
            # Add results to SummaryWriter
            writer.add_scalars(main_tag="Loss", 
                               tag_scalar_dict={"train_loss": train_loss,
                                                "test_loss": test_loss},
                               global_step=epoch)
            writer.add_scalars(main_tag="Accuracy", 
                               tag_scalar_dict={"train_acc": train_acc,
                                                "test_acc": test_acc}, 
                               global_step=epoch)

            # Close the writer
            writer.close()
        else:
            pass
    ### End new ###

    # Return the filled results at the end of the epochs
    return results

### Download data

Using the same data from https://www.learnpytorch.io/07_pytorch_experiment_tracking/

In [9]:
# Download 10 percent and 20 percent training data (if necessary)
data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                                     destination="pizza_steak_sushi")

data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

[INFO] data\pizza_steak_sushi directory exists, skipping download.
[INFO] data\pizza_steak_sushi_20_percent directory exists, skipping download.


In [10]:
# Setup training directory paths
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"

# Setup testing directory paths (note: use the same test dataset for both to compare the results)
test_dir = data_10_percent_path / "test"

# Check the directories
print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Testing directory: {test_dir}")

Training directory 10%: data\pizza_steak_sushi\train
Training directory 20%: data\pizza_steak_sushi_20_percent\train
Testing directory: data\pizza_steak_sushi\test


In [11]:
from torchvision import transforms

# Create a transform to normalize data distribution to be inline with ImageNet
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], # values per colour channel [red, green, blue]
                                 std=[0.229, 0.224, 0.225])

# Create a transform pipeline
simple_transform = transforms.Compose([
                                       transforms.Resize((224, 224)),
                                       transforms.ToTensor(), # get image values between 0 & 1
                                       normalize
])

### Turn data into DataLoaders 

In [12]:
BATCH_SIZE = 32

# Create 10% training and test DataLoaders
train_dataloader_10_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_10_percent,
                                                                                          test_dir=test_dir,
                                                                                          transform=simple_transform,
                                                                                          batch_size=BATCH_SIZE)

# Create 20% training and test DataLoaders
train_dataloader_20_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_20_percent,
                                                                                          test_dir=test_dir,
                                                                                          transform=simple_transform,
                                                                                          batch_size=BATCH_SIZE)

# Find the number of samples/batches per dataloader (using the same test_dataloader for both experiments)
print(f"Number of batches of size {BATCH_SIZE} in 10 percent training data: {len(train_dataloader_10_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in 20 percent training data: {len(train_dataloader_20_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in testing data: {len(train_dataloader_10_percent)} (all experiments will use the same test set)")
print(f"Number of classes: {len(class_names)}, class names: {class_names}")

Number of batches of size 32 in 10 percent training data: 8
Number of batches of size 32 in 20 percent training data: 15
Number of batches of size 32 in testing data: 8 (all experiments will use the same test set)
Number of classes: 3, class names: ['pizza', 'steak', 'sushi']


## Exercise 1: Pick a larger model from [`torchvision.models`](https://pytorch.org/vision/main/models.html) to add to the list of experiments (for example, EffNetB3 or higher)

* How does it perform compared to our existing models?
* **Hint:** You'll need to set up an exerpiment similar to [07. PyTorch Experiment Tracking section 7.6](https://www.learnpytorch.io/07_pytorch_experiment_tracking/#76-create-experiments-and-set-up-training-code).

In [13]:
# Exercise 1: add a larger model (EffNetB3) to the list of experiments
weights_b3 = torchvision.models.EfficientNet_B3_Weights.DEFAULT
auto_transforms_b3 = weights_b3.transforms()

train_dataloader_10_percent_b3, test_dataloader_b3, class_names = data_setup.create_dataloaders(
    train_dir=train_dir_10_percent,
    test_dir=test_dir,
    transform=auto_transforms_b3,
    batch_size=BATCH_SIZE
)

def create_effnetb3():
  """Creates an EffNetB3 feature extractor model with a frozen base and a fresh classifier head."""
  weights = torchvision.models.EfficientNet_B3_Weights.DEFAULT
  model = torchvision.models.efficientnet_b3(weights=weights).to(device)

  for param in model.features.parameters():
    param.requires_grad = False

  set_seeds()
  model.classifier = nn.Sequential(
      nn.Dropout(p=0.3, inplace=True),
      nn.Linear(in_features=1536, out_features=len(class_names))
  ).to(device)
  model.name = "effnetb3"
  print(f"[INFO] Created new {model.name} model.")
  return model

set_seeds()
effnetb3 = create_effnetb3()

writer = create_writer(experiment_name="10_percent_data",
                       model_name="effnetb3",
                       extra="5_epochs")

effnetb3_results = train(model=effnetb3,
                         train_dataloader=train_dataloader_10_percent_b3,
                         test_dataloader=test_dataloader_b3,
                         optimizer=torch.optim.Adam(effnetb3.parameters(), lr=0.001),
                         loss_fn=nn.CrossEntropyLoss(),
                         epochs=5,
                         device=device,
                         writer=writer)

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to C:\Users\Admin/.cache\torch\hub\checkpoints\efficientnet_b3_rwightman-b3899882.pth


  0%|          | 0.00/47.2M [00:00<?, ?B/s]

  9%|▊         | 4.12M/47.2M [00:00<00:01, 43.2MB/s]

 24%|██▍       | 11.2M/47.2M [00:00<00:00, 61.3MB/s]

 40%|███▉      | 18.8M/47.2M [00:00<00:00, 68.7MB/s]

...

 80%|████████  | 4/5 [08:15<02:02, 122.32s/it]

Epoch: 4 | train_loss: 0.7141 | train_acc: 0.9023 | test_loss: 0.6623 | test_acc: 0.9167


100%|██████████| 5/5 [09:59<00:00, 115.87s/it]

100%|██████████| 5/5 [09:59<00:00, 119.94s/it]

Epoch: 5 | train_loss: 0.7119 | train_acc: 0.7383 | test_loss: 0.6321 | test_acc: 0.8854


In [14]:
# Compare EffNetB3 to the smaller models tried earlier in this notebook (if their results exist)
import pandas as pd

all_results = {"effnetb3 (10%, 5 epochs)": {k: v[-1] for k, v in effnetb3_results.items()}}
for name in ["effnetb0_results", "effnetb2_results"]:
  if name in globals():
    all_results[name.replace("_results", "")] = {k: v[-1] for k, v in globals()[name].items()}

pd.DataFrame(all_results).T

,train_loss,train_acc,test_loss,test_acc
"effnetb3 (10%, 5 epochs)",0.711939,0.738281,0.632141,0.885417


**Nhận xét:** EffNetB3 có nhiều tham số hơn (và dùng ảnh đầu vào lớn hơn, do `weights.transforms()` tự quyết định
kích thước phù hợp với mô hình) nên có khả năng học các đặc trưng phức tạp hơn EffNetB0/B2, nhưng đổi lại thời
gian huấn luyện/suy luận chậm hơn đáng kể trên CPU. Với tập dữ liệu nhỏ (10%) sự khác biệt về độ chính xác có thể
không lớn — mô hình lớn hơn phát huy ưu thế rõ nhất khi có đủ dữ liệu để tận dụng.

## Exercise 2. Introduce data augmentation to the list of experiments using the 20% pizza, steak, sushi training and test datasets, does this change anything?
    
* For example, you could have one training DataLoader that uses data augmentation (e.g. `train_dataloader_20_percent_aug` and `train_dataloader_20_percent_no_aug`) and then compare the results of two of the same model types training on these two DataLoaders.
* **Note:** You may need to alter the `create_dataloaders()` function to be able to take a transform for the training data and the testing data (because you don't need to perform data augmentation on the test data). See [04. PyTorch Custom Datasets section 6](https://www.learnpytorch.io/04_pytorch_custom_datasets/#6-other-forms-of-transforms-data-augmentation) for examples of using data augmentation or the script below for an example:

```python
# Note: Data augmentation transform like this should only be performed on training data
train_transform_data_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    normalize
])

# Create a helper function to visualize different augmented (and not augmented) images
def view_dataloader_images(dataloader, n=10):
    if n > 10:
        print(f"Having n higher than 10 will create messy plots, lowering to 10.")
        n = 10
    imgs, labels = next(iter(dataloader))
    plt.figure(figsize=(16, 8))
    for i in range(n):
        # Min max scale the image for display purposes
        targ_image = imgs[i]
        sample_min, sample_max = targ_image.min(), targ_image.max()
        sample_scaled = (targ_image - sample_min)/(sample_max - sample_min)

        # Plot images with appropriate axes information
        plt.subplot(1, 10, i+1)
        plt.imshow(sample_scaled.permute(1, 2, 0)) # resize for Matplotlib requirements
        plt.title(class_names[labels[i]])
        plt.axis(False)

# Have to update `create_dataloaders()` to handle different augmentations
import os
from torch.utils.data import DataLoader
from torchvision import datasets

NUM_WORKERS = os.cpu_count() # use maximum number of CPUs for workers to load data 

# Note: this is an update version of data_setup.create_dataloaders to handle
# differnt train and test transforms.
def create_dataloaders(
    train_dir, 
    test_dir, 
    train_transform, # add parameter for train transform (transforms on train dataset)
    test_transform,  # add parameter for test transform (transforms on test dataset)
    batch_size=32, num_workers=NUM_WORKERS
):
    # Use ImageFolder to create dataset(s)
    train_data = datasets.ImageFolder(train_dir, transform=train_transform)
    test_data = datasets.ImageFolder(test_dir, transform=test_transform)

    # Get class names
    class_names = train_data.classes

    # Turn images into data loaders
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names
```

In [15]:
# Exercise 2: does data augmentation help on the 20% pizza, steak, sushi dataset?
import os
from torch.utils.data import DataLoader
from torchvision import datasets

NUM_WORKERS = 0 if os.name == "nt" else os.cpu_count()

def create_dataloaders_v2(train_dir, test_dir, train_transform, test_transform,
                          batch_size=32, num_workers=NUM_WORKERS):
  """Like data_setup.create_dataloaders but allows different train/test transforms
  (test data should not be augmented)."""
  train_data = datasets.ImageFolder(train_dir, transform=train_transform)
  test_data = datasets.ImageFolder(test_dir, transform=test_transform)
  class_names = train_data.classes
  train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
  test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=num_workers)
  return train_dataloader, test_dataloader, class_names

# No augmentation: same transform used everywhere else in this notebook
train_transform_no_aug = simple_transform

# Augmentation: only ever applied to training data
train_transform_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    normalize
])

train_dataloader_20_percent_no_aug, test_dataloader_20, class_names = create_dataloaders_v2(
    train_dir=train_dir_20_percent, test_dir=test_dir,
    train_transform=train_transform_no_aug, test_transform=simple_transform,
    batch_size=BATCH_SIZE)

train_dataloader_20_percent_aug, _, _ = create_dataloaders_v2(
    train_dir=train_dir_20_percent, test_dir=test_dir,
    train_transform=train_transform_aug, test_transform=simple_transform,
    batch_size=BATCH_SIZE)

def create_effnetb0():
  weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
  model = torchvision.models.efficientnet_b0(weights=weights).to(device)
  for param in model.features.parameters():
    param.requires_grad = False
  set_seeds()
  model.classifier = nn.Sequential(
      nn.Dropout(p=0.2, inplace=True),
      nn.Linear(in_features=1280, out_features=len(class_names))
  ).to(device)
  model.name = "effnetb0"
  return model

# Train without augmentation
set_seeds()
model_no_aug = create_effnetb0()
writer_no_aug = create_writer("data_20_percent", "effnetb0", "5_epochs_no_aug")
results_no_aug = train(model=model_no_aug,
                       train_dataloader=train_dataloader_20_percent_no_aug,
                       test_dataloader=test_dataloader_20,
                       optimizer=torch.optim.Adam(model_no_aug.parameters(), lr=0.001),
                       loss_fn=nn.CrossEntropyLoss(),
                       epochs=5,
                       device=device,
                       writer=writer_no_aug)

# Train with augmentation
set_seeds()
model_aug = create_effnetb0()
writer_aug = create_writer("data_20_percent", "effnetb0", "5_epochs_aug")
results_aug = train(model=model_aug,
                    train_dataloader=train_dataloader_20_percent_aug,
                    test_dataloader=test_dataloader_20,
                    optimizer=torch.optim.Adam(model_aug.parameters(), lr=0.001),
                    loss_fn=nn.CrossEntropyLoss(),
                    epochs=5,
                    device=device,
                    writer=writer_aug)

[INFO] Created SummaryWriter, saving to: runs\2026-09-24\data_20_percent\effnetb0\5_epochs_no_aug...


  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:42<02:49, 42.34s/it]

Epoch: 1 | train_loss: 0.9679 | train_acc: 0.5583 | test_loss: 0.6605 | test_acc: 0.8655


 40%|████      | 2/5 [01:18<01:56, 38.83s/it]

...

 80%|████████  | 4/5 [02:44<00:41, 41.12s/it]

Epoch: 4 | train_loss: 0.5280 | train_acc: 0.8417 | test_loss: 0.4254 | test_acc: 0.9072


100%|██████████| 5/5 [03:32<00:00, 43.40s/it]

100%|██████████| 5/5 [03:32<00:00, 42.45s/it]

Epoch: 5 | train_loss: 0.5481 | train_acc: 0.8396 | test_loss: 0.4156 | test_acc: 0.9384


In [16]:
# Compare augmented vs non-augmented training
comparison_df = pd.DataFrame({
    "no augmentation": {k: v[-1] for k, v in results_no_aug.items()},
    "with augmentation (TrivialAugmentWide)": {k: v[-1] for k, v in results_aug.items()},
}).T
comparison_df

,train_loss,train_acc,test_loss,test_acc
no augmentation,0.397604,0.912500,0.390595,0.917614
with augmentation (TrivialAugmentWide),0.548137,0.839583,0.415573,0.938447


**Nhận xét:** với một tập dữ liệu nhỏ như 20% pizza/steak/sushi và chỉ 5 epoch, data augmentation thường làm
`train_acc` **giảm nhẹ** (vì bài toán học khó hơn — mô hình không được nhìn ảnh gốc mà luôn nhìn phiên bản biến
đổi ngẫu nhiên) nhưng có thể giúp `test_acc` ổn định hơn hoặc nhỉnh hơn nhờ giảm overfitting. Augmentation phát
huy tác dụng rõ nhất khi huấn luyện lâu hơn (nhiều epoch hơn) trên tập dữ liệu nhỏ, chứ 5 epoch có thể là chưa đủ
để thấy lợi ích rõ rệt.

## Exercise 3. Scale up the dataset to turn FoodVision Mini into FoodVision Big using the entire [Food101 dataset from `torchvision.models`](https://pytorch.org/vision/stable/generated/torchvision.datasets.Food101.html#torchvision.datasets.Food101)
    
* You could take the best performing model from your various experiments or even the EffNetB2 feature extractor we created in this notebook and see how it goes fitting for 5 epochs on all of Food101.
* If you try more than one model, it would be good to have the model's results tracked.
* If you load the Food101 dataset from `torchvision.models`, you'll have to create PyTorch DataLoaders to use it in training.
* **Note:** Due to the larger amount of data in Food101 compared to our pizza, steak, sushi dataset, this model will take longer to train.

In [17]:
# Exercise 3: scale up to the full Food101 dataset (101 classes, ~101,000 images, ~5GB download).
# NOTE: this is very slow to train on CPU (can take many hours per epoch), so the full
# pipeline is written below but is NOT executed automatically in this run.
# Set RUN_FOODVISION_BIG = True and re-run this cell (ideally on a machine with a GPU)
# to actually download the data and train the model.
RUN_FOODVISION_BIG = False

if RUN_FOODVISION_BIG:
  weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
  auto_transforms = weights.transforms()

  # torchvision.datasets.Food101 downloads and extracts the full dataset automatically
  train_data_food101 = datasets.Food101(root="data",
                                        split="train",
                                        transform=auto_transforms,
                                        download=True)
  test_data_food101 = datasets.Food101(root="data",
                                       split="test",
                                       transform=auto_transforms,
                                       download=True)
  food101_class_names = train_data_food101.classes
  print(f"[INFO] Food101: {len(train_data_food101)} train images, "
        f"{len(test_data_food101)} test images, {len(food101_class_names)} classes")

  BATCH_SIZE_FOOD101 = 32
  train_dataloader_food101 = DataLoader(train_data_food101,
                                        batch_size=BATCH_SIZE_FOOD101,
                                        shuffle=True,
                                        num_workers=NUM_WORKERS)
  test_dataloader_food101 = DataLoader(test_data_food101,
                                       batch_size=BATCH_SIZE_FOOD101,
                                       shuffle=False,
                                       num_workers=NUM_WORKERS)

  model_food101 = torchvision.models.efficientnet_b2(weights=weights).to(device)
  for param in model_food101.features.parameters():
    param.requires_grad = False

  set_seeds()
  model_food101.classifier = nn.Sequential(
      nn.Dropout(p=0.3, inplace=True),
      nn.Linear(in_features=1408, out_features=len(food101_class_names))
  ).to(device)

  writer = create_writer(experiment_name="food101_full",
                         model_name="effnetb2",
                         extra="5_epochs")

  food101_results = train(model=model_food101,
                          train_dataloader=train_dataloader_food101,
                          test_dataloader=test_dataloader_food101,
                          optimizer=torch.optim.Adam(model_food101.parameters(), lr=0.001),
                          loss_fn=nn.CrossEntropyLoss(),
                          epochs=5,
                          device=device,
                          writer=writer)
else:
  print("[INFO] RUN_FOODVISION_BIG=False -- full Food101 training code above was NOT executed")
  print("[INFO] (101 classes, ~5GB download, very slow to train on CPU).")
  print("[INFO] Set RUN_FOODVISION_BIG = True above and re-run this cell to train for real.")

[INFO] RUN_FOODVISION_BIG=False -- full Food101 training code above was NOT executed
[INFO] (101 classes, ~5GB download, very slow to train on CPU).
[INFO] Set RUN_FOODVISION_BIG = True above and re-run this cell to train for real.
